## Term Sheet Reconciliation Workflow

Following your preferences, this notebook builds an end-to-end GenAI reconciliation pipeline for the assignment:

* Reads the assignment term sheets and booking extracts from `/Volumes/vabtp_dev/dbfs_files/vabtp_dev/PDF_EXTRACTION/Assignment/`
* Parses term sheets with `ai_parse_document`
* Exports extracted PDF tables to Excel files under `Term-Sheets-Extracts`
* Uses an LLM pass to extract structured fields plus evidence snippets
* Validates extracted fields against a strict typed schema
* Ingests the CSV and JSON booking files
* Runs reconciliation and a second LLM pass for mismatch adjudication
* Saves final artifacts under `Reconciliation-Outputs`

The notebook is designed so the main flow performs extraction first and then immediately continues with reconciliation steps in the same execution path.

In [0]:
%pip install openpyxl lxml --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
import io
import re
import json
from datetime import datetime
from decimal import Decimal, InvalidOperation
from typing import Any, Dict, List, Optional

import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Border, Side
from pyspark.sql import functions as F
from pyspark.sql.types import *

ASSIGNMENT_BASE = "/Volumes/vabtp_dev/dbfs_files/vabtp_dev/PDF_EXTRACTION/Assignment"
TERM_SHEET_OUTPUT_DIR = f"{ASSIGNMENT_BASE}/Term-Sheets-Extracts"
RECON_OUTPUT_DIR = f"{ASSIGNMENT_BASE}/Reconciliation-Outputs"
TMP_IMG_DIR = "/tmp/assignment_term_sheet_images"

os.makedirs(TERM_SHEET_OUTPUT_DIR, exist_ok=True)
os.makedirs(RECON_OUTPUT_DIR, exist_ok=True)
os.makedirs(TMP_IMG_DIR, exist_ok=True)

TERM_SHEETS = {
    "idbi": f"{ASSIGNMENT_BASE}/3.Term Sheet - IDBI.pdf",
    "genel": f"{ASSIGNMENT_BASE}/4.Term Sheet - Genel Energy.pdf",
}

BOOKING_FILES = {
    "idbi": {
        "csv": f"{ASSIGNMENT_BASE}/IDBI_Omni.csv",
        "json": f"{ASSIGNMENT_BASE}/IDBI_Omni.json",
    },
    "genel": {
        "csv": f"{ASSIGNMENT_BASE}/Genel_Energy.csv",
        "json": f"{ASSIGNMENT_BASE}/Genel_Energy.json",
    },
}

TARGET_FIELDS = [
    "isin", "issuer", "coupon", "currency", "maturity", "issue_date", "settlement_date",
    "issue_amount", "issue_price", "nominal_amount_per_bond", "interest_payment_date",
    "interest_payment_frequency", "day_count_fraction", "business_day_convention",
    "business_day_location", "amortization_type", "minimum_subscription", "parent", "notional"
]

print("Setup complete")
print(f"Assignment base      : {ASSIGNMENT_BASE}")
print(f"Excel extract output : {TERM_SHEET_OUTPUT_DIR}")
print(f"Reconciliation output: {RECON_OUTPUT_DIR}")

Setup complete
Assignment base      : /Volumes/vabtp_dev/dbfs_files/vabtp_dev/PDF_EXTRACTION/Assignment
Excel extract output : /Volumes/vabtp_dev/dbfs_files/vabtp_dev/PDF_EXTRACTION/Assignment/Term-Sheets-Extracts
Reconciliation output: /Volumes/vabtp_dev/dbfs_files/vabtp_dev/PDF_EXTRACTION/Assignment/Reconciliation-Outputs


In [0]:
class EvidenceBundle(BaseModel):
    isin: Optional[str] = None
    issuer: Optional[str] = None
    coupon: Optional[str] = None
    currency: Optional[str] = None
    maturity: Optional[str] = None
    issue_date: Optional[str] = None
    settlement_date: Optional[str] = None
    issue_amount: Optional[str] = None
    issue_price: Optional[str] = None
    nominal_amount_per_bond: Optional[str] = None
    interest_payment_date: Optional[str] = None
    interest_payment_frequency: Optional[str] = None
    day_count_fraction: Optional[str] = None
    business_day_convention: Optional[str] = None
    business_day_location: Optional[str] = None
    amortization_type: Optional[str] = None
    minimum_subscription: Optional[str] = None
    parent: Optional[str] = None
    notional: Optional[str] = None


class ExtractionSchema(BaseModel):
    isin: Optional[str] = None
    issuer: Optional[str] = None
    coupon: Optional[float] = None
    currency: Optional[str] = None
    maturity: Optional[str] = None
    issue_date: Optional[str] = None
    settlement_date: Optional[str] = None
    issue_amount: Optional[float] = None
    issue_price: Optional[float] = None
    nominal_amount_per_bond: Optional[float] = None
    interest_payment_date: Optional[str] = None
    interest_payment_frequency: Optional[str] = None
    day_count_fraction: Optional[str] = None
    business_day_convention: Optional[str] = None
    business_day_location: Optional[List[str]] = None
    amortization_type: Optional[str] = None
    minimum_subscription: Optional[float] = None
    parent: Optional[str] = None
    notional: Optional[float] = None
    evidence: EvidenceBundle


FIELD_NAME_MAP = {
    "ISIN": "isin",
    "Issuer": "issuer",
    "Coupon": "coupon",
    "Currency": "currency",
    "Maturity": "maturity",
    "IssueDate": "issue_date",
    "SettlementDate": "settlement_date",
    "IssueAmount": "issue_amount",
    "IssuePrice": "issue_price",
    "NominalAmountPerBond": "nominal_amount_per_bond",
    "InterestPaymentDate": "interest_payment_date",
    "InterestPaymentFrequency": "interest_payment_frequency",
    "DayCountFraction": "day_count_fraction",
    "BusinessDayConvention": "business_day_convention",
    "BusinessDayLocation": "business_day_location",
    "AmortizationType": "amortization_type",
    "MinimumSubscription": "minimum_subscription",
    "Parent": "parent",
    "Notional": "notional",
}

NUMERIC_FIELDS = {"coupon", "issue_amount", "issue_price", "nominal_amount_per_bond", "minimum_subscription", "notional"}
DATE_FIELDS = {"issue_date", "settlement_date", "interest_payment_date"}


def extract_json_block(text: str) -> Dict[str, Any]:
    cleaned = (text or "").strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
    return json.loads(cleaned)


def normalize_whitespace(value: Optional[str]) -> Optional[str]:
    if value is None:
        return None
    value = str(value).replace("\n", " ").strip()
    value = re.sub(r"\s+", " ", value)
    return value or None


def normalize_isin(value: Optional[str]) -> Optional[str]:
    value = normalize_whitespace(value)
    if value is None:
        return None
    return value.replace(" ", "")


def parse_numeric(value: Any) -> Optional[float]:
    if value is None or value == "":
        return None
    if isinstance(value, (int, float)):
        return float(value)
    text = normalize_whitespace(str(value))
    if text is None:
        return None
    lowered = text.lower()
    multiplier = 1.0
    if "crore" in lowered:
        multiplier = 10_000_000.0
    elif "million" in lowered:
        multiplier = 1_000_000.0
    elif "billion" in lowered:
        multiplier = 1_000_000_000.0

    numeric_candidates = re.findall(r"\d[\d,./-]*\d|\d", text)
    if not numeric_candidates:
        return None
    numeric_text = numeric_candidates[0]
    numeric_text = numeric_text.replace(",", "")
    numeric_text = numeric_text.replace("/-", "")
    numeric_text = numeric_text.replace("/", "")
    if numeric_text.count(".") > 1:
        numeric_text = numeric_text.split(".")[0]
    numeric_text = re.sub(r"[^0-9.\-]", "", numeric_text)
    if numeric_text in {"", ".", "-"}:
        return None
    return float(Decimal(numeric_text) * Decimal(str(multiplier)))


def parse_date_to_iso(value: Any) -> Optional[str]:
    text = normalize_whitespace(value)
    if text is None:
        return None
    text = text.replace("Expected to be ", "")
    text = re.sub(r"\(.*?\)", "", text).strip()
    for fmt in ["%Y-%m-%d", "%d %B %Y", "%d %b %Y", "%B %d, %Y", "%d/%m/%Y", "%m/%d/%Y"]:
        try:
            return datetime.strptime(text, fmt).date().isoformat()
        except ValueError:
            continue
    return text


def normalize_text_field(field: str, value: Any) -> Any:
    if field == "isin":
        return normalize_isin(value)
    if field in NUMERIC_FIELDS:
        return parse_numeric(value)
    if field in DATE_FIELDS:
        return parse_date_to_iso(value)
    if field == "business_day_location":
        if value is None:
            return None
        if isinstance(value, list):
            return [normalize_whitespace(v) for v in value if normalize_whitespace(v)]
        text = normalize_whitespace(value)
        if text is None:
            return None
        parts = re.split(r"\||,| and ", text)
        return [normalize_whitespace(v) for v in parts if normalize_whitespace(v)]
    return normalize_whitespace(value)


def coerce_extraction_payload(doc_name: str, raw_payload: Dict[str, Any]) -> Dict[str, Any]:
    payload = {field: raw_payload.get(field) for field in TARGET_FIELDS}
    payload["evidence"] = raw_payload.get("evidence", {})
    payload["isin"] = normalize_isin(payload.get("isin"))
    if doc_name == "genel" and payload.get("settlement_date") is None:
        payload["settlement_date"] = payload.get("issue_date")
    if doc_name == "idbi" and payload.get("amortization_type") is None:
        payload["amortization_type"] = "Perpetual"
    if doc_name == "idbi" and payload.get("currency") is None:
        payload["currency"] = "INR"
    for field in TARGET_FIELDS:
        payload[field] = normalize_text_field(field, payload.get(field))
    if doc_name == "idbi" and payload.get("minimum_subscription") is not None and payload.get("nominal_amount_per_bond") is not None:
        minimum_subscription_text = normalize_whitespace(raw_payload.get("minimum_subscription")) or ""
        bond_count_match = re.search(r"(\d+)\s+Bond", minimum_subscription_text, flags=re.IGNORECASE)
        if bond_count_match:
            payload["minimum_subscription"] = float(bond_count_match.group(1)) * float(payload["nominal_amount_per_bond"])
    if payload.get("issuer") and doc_name == "genel":
        payload["issuer"] = re.sub(r"\s*\(.*", "", payload["issuer"]).strip()
    if payload.get("parent"):
        payload["parent"] = re.sub(r"\s*\(.*", "", payload["parent"]).strip()
    if isinstance(payload.get("amortization_type"), str) and "repaid in full" in payload["amortization_type"].lower():
        payload["amortization_type"] = "Bullet"
    return payload


def field_equal(field: str, extracted_value: Any, booking_value: Any) -> bool:
    left = normalize_text_field(field, extracted_value)
    right = normalize_text_field(field, booking_value)
    if left is None and right is None:
        return True
    if field in NUMERIC_FIELDS:
        if left is None or right is None:
            return False
        return abs(float(left) - float(right)) < 1e-9
    if field == "business_day_location":
        return left == right
    return left == right


print("Schema and helper functions ready")

Schema and helper functions ready


In [0]:
pdf_parse_df = spark.sql(f"""
WITH docs AS (
  SELECT
    path,
    content,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed
  FROM READ_FILES('{ASSIGNMENT_BASE}/', format => 'binaryFile')
  WHERE path LIKE '%3.Term Sheet - IDBI.pdf' OR path LIKE '%4.Term Sheet - Genel Energy.pdf'
)
SELECT
  path,
  parsed,
  transform(
    filter(try_cast(parsed:document:elements AS ARRAY<VARIANT>), x -> try_cast(x:type AS STRING) = 'table'),
    x -> try_cast(x:content AS STRING)
  ) AS table_html_list,
  concat_ws('\n\n', transform(try_cast(parsed:document:elements AS ARRAY<VARIANT>), x -> try_cast(x:content AS STRING))) AS full_text
FROM docs
WHERE try_cast(parsed:error_status AS STRING) IS NULL
ORDER BY path
""")

pdf_records = [row.asDict(recursive=True) for row in pdf_parse_df.collect()]

header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
header_font = Font(color='FFFFFF', bold=True)
thin = Side(style='thin')
all_border = Border(left=thin, right=thin, top=thin, bottom=thin)

excel_exports = []
for rec in pdf_records:
    file_name = os.path.basename(rec['path']).replace('.pdf', '.xlsx')
    output_path = f"{TERM_SHEET_OUTPUT_DIR}/{file_name}"
    wb = Workbook()
    ws = wb.active
    ws.title = 'Extracted Tables'
    current_row = 1
    ws.cell(current_row, 1, 'Source Path')
    ws.cell(current_row, 2, rec['path'])
    current_row += 2

    for idx, table_html in enumerate(rec['table_html_list'], start=1):
        try:
            table_df = pd.read_html(io.StringIO(table_html))[0].fillna('')
        except ValueError:
            continue
        ws.cell(current_row, 1, f'Table {idx}')
        ws.cell(current_row, 1).font = Font(bold=True)
        current_row += 1
        for col_idx, column_name in enumerate(table_df.columns.tolist(), start=1):
            cell = ws.cell(current_row, col_idx, str(column_name))
            cell.fill = header_fill
            cell.font = header_font
            cell.border = all_border
        current_row += 1
        for row_values in table_df.itertuples(index=False):
            for col_idx, value in enumerate(row_values, start=1):
                cell = ws.cell(current_row, col_idx, str(value))
                cell.border = all_border
            current_row += 1
        current_row += 2
    local_tmp = f"/tmp/{file_name}"
    wb.save(local_tmp)
    with open(local_tmp, 'rb') as src, open(output_path, 'wb') as dst:
        dst.write(src.read())
    excel_exports.append({"pdf_path": rec['path'], "excel_path": output_path, "table_count": len(rec['table_html_list'])})

excel_exports_df = pd.DataFrame(excel_exports)
display(excel_exports_df)
print(f"Exported {len(excel_exports_df)} Excel files")

pdf_path,excel_path,table_count
dbfs:/Volumes/vabtp_dev/dbfs_files/vabtp_dev/PDF_EXTRACTION/Assignment/3.Term Sheet - IDBI.pdf,/Volumes/vabtp_dev/dbfs_files/vabtp_dev/PDF_EXTRACTION/Assignment/Term-Sheets-Extracts/3.Term Sheet - IDBI.xlsx,15
dbfs:/Volumes/vabtp_dev/dbfs_files/vabtp_dev/PDF_EXTRACTION/Assignment/4.Term Sheet - Genel Energy.pdf,/Volumes/vabtp_dev/dbfs_files/vabtp_dev/PDF_EXTRACTION/Assignment/Term-Sheets-Extracts/4.Term Sheet - Genel Energy.xlsx,9


Exported 2 Excel files


In [0]:
llm_requests_df = pdf_parse_df.select('path', 'full_text')
llm_requests_df.createOrReplaceTempView('assignment_term_sheet_text')

extraction_prompt = """
Return JSON only. Extract these fields from the term sheet text:
isin, issuer, coupon, currency, maturity, issue_date, settlement_date, issue_amount, issue_price,
nominal_amount_per_bond, interest_payment_date, interest_payment_frequency, day_count_fraction,
business_day_convention, business_day_location, amortization_type, minimum_subscription, parent, notional.
Requirements:
1. evidence must be an object mapping every field to the exact source quote or null.
2. Keep dates as written in the document.
3. Keep amounts and percentages exactly as written in the document.
4. If a value is not explicitly present, return null.
"""

llm_raw_df = spark.sql(f"""
SELECT
  path,
  ai_query(
    'databricks-claude-sonnet-4',
    '{extraction_prompt.strip().replace("'", "''")}' || '\n\nDocument text:\n' || full_text
  ) AS extraction_text
FROM assignment_term_sheet_text
ORDER BY path
""")

validated_rows = []
for row in llm_raw_df.collect():
    path = row['path']
    doc_name = 'idbi' if 'IDBI' in path else 'genel'
    raw_text = row['extraction_text']
    parse_status = 'validated'
    parsing_error = None
    normalized_payload = None
    try:
        raw_payload = extract_json_block(raw_text)
        normalized_payload = coerce_extraction_payload(doc_name, raw_payload)
        validated = ExtractionSchema.model_validate(normalized_payload)
        validated_rows.append({
            'document_key': doc_name,
            'pdf_path': path,
            'parse_status': parse_status,
            'parsing_error': parsing_error,
            'validated_payload_json': validated.model_dump_json(),
            'raw_extraction_text': raw_text,
        })
    except (json.JSONDecodeError, ValidationError, TypeError, InvalidOperation) as exc:
        parse_status = 'parsing_error'
        parsing_error = str(exc)
        validated_rows.append({
            'document_key': doc_name,
            'pdf_path': path,
            'parse_status': parse_status,
            'parsing_error': parsing_error,
            'validated_payload_json': json.dumps(normalized_payload or {}),
            'raw_extraction_text': raw_text,
        })

validated_extractions_df = pd.DataFrame(validated_rows)
display(validated_extractions_df)
print(validated_extractions_df[['document_key', 'parse_status']])

document_key,pdf_path,parse_status,parsing_error,validated_payload_json,raw_extraction_text
idbi,dbfs:/Volumes/vabtp_dev/dbfs_files/vabtp_dev/PDF_EXTRACTION/Assignment/3.Term Sheet - IDBI.pdf,validated,null,"{""isin"":null,""issuer"":""IDBI Bank Limited"",""coupon"":10.75,""currency"":""INR"",""maturity"":""Perpetual"",""issue_date"":""2014-09-29"",""settlement_date"":""2014-10-17"",""issue_amount"":15000000000.0,""issue_price"":null,""nominal_amount_per_bond"":1000000.0,""interest_payment_date"":""On the anniversary of the Deemed Date of Allotment"",""interest_payment_frequency"":""annually in arrear"",""day_count_fraction"":""Actual/Actual"",""business_day_convention"":""Should any of the dates, other than the Coupon Payment Date including the Deemed Date of Allotment, Issuer Call Date, Tax Call Date or Regulatory Call Date as defined in this Information Memorandum, fall on day which is not a business day, the immediately preceding business day shall be considered as the effective date. Should the Coupon Payment Date, as defined in this Disclosure Document, fall on day which is not a business day, the immediately next business day shall be considered as the effective date"",""business_day_location"":null,""amortization_type"":""Perpetual"",""minimum_subscription"":5000000.0,""parent"":null,""notional"":null,""evidence"":{""isin"":null,""issuer"":""IDBI Bank Limited"",""coupon"":""10.75% p.a."",""currency"":null,""maturity"":""Perpetual"",""issue_date"":""September 29, 2014"",""settlement_date"":""October 17, 2014"",""issue_amount"":""Rs.1,500 Crores with an option to retain over subscription up to Rs.1,000 Crores"",""issue_price"":null,""nominal_amount_per_bond"":""Rs.10,00,000/- (Rupees Ten Lakh) per Bond"",""interest_payment_date"":""On the anniversary of the Deemed Date of Allotment"",""interest_payment_frequency"":""annually in arrear"",""day_count_fraction"":""Actual/Actual"",""business_day_convention"":""Should any of the dates, other than the Coupon Payment Date including the Deemed Date of Allotment, Issuer Call Date, Tax Call Date or Regulatory Call Date as defined in this Information Memorandum, fall on day which is not a business day, the immediately preceding business day shall be considered as the effective date. Should the Coupon Payment Date, as defined in this Disclosure Document, fall on day which is not a business day, the immediately next business day shall be considered as the effective date"",""business_day_location"":null,""amortization_type"":null,""minimum_subscription"":""5 Bonds and in multiples of 1 Bond thereafter"",""parent"":null,""notional"":null}}","```json { ""isin"": null, ""issuer"": ""IDBI Bank Limited"", ""coupon"": ""10.75% p.a."", ""currency"": null, ""maturity"": ""Perpetual"", ""issue_date"": ""September 29, 2014"", ""settlement_date"": ""October 17, 2014"", ""issue_amount"": ""Rs.1,500 Crores with an option to retain over subscription up to Rs.1,000 Crores"", ""issue_price"": null, ""nominal_amount_per_bond"": ""Rs.10,00,000/- (Rupees Ten Lakh) per Bond"", ""interest_payment_date"": ""On the anniversary of the Deemed Date of Allotment"", ""interest_payment_frequency"": ""annually in arrear"", ""day_count_fraction"": ""Actual/Actual"", ""business_day_convention"": ""Should any of the dates, other than the Coupon Payment Date including the Deemed Date of Allotment, Issuer Call Date, Tax Call Date or Regulatory Call Date as defined in this Information Memorandum, fall on day which is not a business day, the immediately preceding business day shall be considered as the effective date. Should the Coupon Payment Date, as defined in this Disclosure Document, fall on day which is not a business day, the immediately next business day shall be considered as the effective date"", ""business_day_location"": null, ""amortization_type"": null, ""minimum_subscription"": ""5 Bonds and in multiples of 1 Bond thereafter"", ""parent"": null, ""notional"": null, ""evidence"": { ""isin"": null, ""issuer"

  document_key parse_status
0         idbi    validated
1        genel    validated


In [0]:
booking_frames = []
for doc_name, files in BOOKING_FILES.items():
    csv_pdf = spark.read.option('header', True).csv(files['csv']).toPandas()
    json_pdf = spark.read.option('multiLine', True).json(files['json']).select(F.explode('trades').alias('trade')).select('trade.*').toPandas()
    csv_pdf['source_format'] = 'csv'
    json_pdf['source_format'] = 'json'
    csv_pdf['document_key'] = doc_name
    json_pdf['document_key'] = doc_name
    booking_frames.append(csv_pdf)
    booking_frames.append(json_pdf)

booking_df = pd.concat(booking_frames, ignore_index=True)

for source_col, target_col in FIELD_NAME_MAP.items():
    if source_col in booking_df.columns:
        booking_df[target_col] = booking_df[source_col].apply(lambda v, tc=target_col: normalize_text_field(tc, v))

validated_payload_lookup = {}
for row in validated_rows:
    if row['parse_status'] == 'validated':
        validated_payload_lookup[row['document_key']] = json.loads(row['validated_payload_json'])

recon_rows = []
trade_id_series = pd.to_numeric(booking_df.get('TradeID'), errors='coerce') if 'TradeID' in booking_df.columns else pd.Series([None] * len(booking_df))
booking_df['trade_id'] = trade_id_series.astype('Int64')
for _, booking_row in booking_df.iterrows():
    doc_name = booking_row['document_key']
    extraction = validated_payload_lookup.get(doc_name)
    if extraction is None:
        recon_rows.append({
            'document_key': doc_name,
            'trade_id': None if pd.isna(booking_row.get('trade_id')) else int(booking_row.get('trade_id')),
            'source_format': booking_row.get('source_format'),
            'field_name': 'all_fields',
            'booking_value': None,
            'extracted_value': None,
            'evidence': None,
            'comparison_result': 'Parsing Error',
            'reason': 'Structured extraction failed validation',
        })
        continue
    for field in TARGET_FIELDS:
        booking_value = booking_row.get(field)
        extracted_value = extraction.get(field)
        evidence_value = extraction.get('evidence', {}).get(field)
        if field_equal(field, extracted_value, booking_value):
            comparison_result = 'Match'
            reason = None
        else:
            comparison_result = 'Mismatch'
            reason = None
        recon_rows.append({
            'document_key': doc_name,
            'trade_id': None if pd.isna(booking_row.get('trade_id')) else int(booking_row.get('trade_id')),
            'source_format': booking_row.get('source_format'),
            'field_name': field,
            'booking_value': booking_value,
            'extracted_value': extracted_value,
            'evidence': evidence_value,
            'comparison_result': comparison_result,
            'reason': reason,
        })

recon_df = pd.DataFrame(recon_rows)
recon_df['booking_value'] = recon_df['booking_value'].apply(lambda v: json.dumps(v) if isinstance(v, list) else (None if v is None else str(v)))
recon_df['extracted_value'] = recon_df['extracted_value'].apply(lambda v: json.dumps(v) if isinstance(v, list) else (None if v is None else str(v)))
recon_df['evidence'] = recon_df['evidence'].apply(lambda v: None if v is None else str(v))
display(recon_df.head(50))
print(recon_df['comparison_result'].value_counts(dropna=False))

document_key,trade_id,source_format,field_name,booking_value,extracted_value,evidence,comparison_result,reason
idbi,1,csv,isin,INE008A08U84,null,null,Mismatch,null
idbi,1,csv,issuer,IDBI Bank Limited,IDBI Bank Limited,IDBI Bank Limited,Match,null
idbi,1,csv,coupon,10.75,10.75,10.75% p.a.,Match,null
idbi,1,csv,currency,INR,INR,null,Match,null
idbi,1,csv,maturity,2025-10-17,Perpetual,Perpetual,Mismatch,null
idbi,1,csv,issue_date,2014-10-17,2014-09-29,"September 29, 2014",Mismatch,null
idbi,1,csv,settlement_date,2014-10-17,2014-10-17,"October 17, 2014",Match,null
idbi,1,csv,issue_amount,15000000000.0,15000000000.0,"Rs.1,500 Crores with an option to retain over subscription up to Rs.1,000 Crores",Match,null
idbi,1,csv,issue_price,100.0,null,null,Mismatch,null
idbi,1,csv,nominal_amount_per_bond,1000000.0,1000000.0,"Rs.10,00,000/- (Rupees Ten Lakh) per Bond",Match,null


comparison_result
Match       231
Mismatch    149
Name: count, dtype: int64


In [0]:
mismatch_df = recon_df[recon_df['comparison_result'] == 'Mismatch'].copy()

if not mismatch_df.empty:
    mismatch_spark_df = spark.createDataFrame(mismatch_df[['document_key', 'trade_id', 'source_format', 'field_name', 'booking_value', 'extracted_value', 'evidence']].astype(str))
    mismatch_spark_df.createOrReplaceTempView('assignment_mismatches')
    adjudication_prompt = """
Return JSON only with keys classification and reason.
Classification must be exactly one of: False Alert, True Mismatch.
A False Alert means the values are semantically equivalent despite formatting or representation differences.
Use the booking value, extracted value, and source evidence.
"""
    adjudicated_df = spark.sql(f"""
    SELECT
      document_key,
      trade_id,
      source_format,
      field_name,
      booking_value,
      extracted_value,
      evidence,
      ai_query(
        'databricks-claude-haiku-4-5',
        '{adjudication_prompt.strip().replace("'", "''")}' ||
        '\nField: ' || field_name ||
        '\nBooking value: ' || booking_value ||
        '\nExtracted value: ' || extracted_value ||
        '\nEvidence: ' || evidence
      ) AS adjudication_text
    FROM assignment_mismatches
    ORDER BY document_key, cast(trade_id as int), source_format, field_name
    """).toPandas()
    adjudicated_df['trade_id'] = pd.to_numeric(adjudicated_df['trade_id'], errors='coerce').astype('Int64')
    recon_df['trade_id'] = pd.to_numeric(recon_df['trade_id'], errors='coerce').astype('Int64')
    adjudicated_df['adjudication_json'] = adjudicated_df['adjudication_text'].apply(extract_json_block)
    adjudicated_df['final_classification'] = adjudicated_df['adjudication_json'].apply(lambda x: x.get('classification'))
    adjudicated_df['final_reason'] = adjudicated_df['adjudication_json'].apply(lambda x: x.get('reason'))
    recon_df = recon_df.merge(
        adjudicated_df[['document_key', 'trade_id', 'source_format', 'field_name', 'final_classification', 'final_reason']],
        on=['document_key', 'trade_id', 'source_format', 'field_name'],
        how='left'
    )
    recon_df['final_status'] = recon_df.apply(
        lambda r: r['final_classification'] if r['comparison_result'] == 'Mismatch' and pd.notna(r['final_classification']) else r['comparison_result'],
        axis=1
    )
    recon_df['reason'] = recon_df.apply(
        lambda r: r['final_reason'] if pd.notna(r.get('final_reason')) else r['reason'],
        axis=1
    )
else:
    recon_df['final_status'] = recon_df['comparison_result']

validated_extractions_path = f"{RECON_OUTPUT_DIR}/validated_extractions.csv"
reconciliation_path = f"{RECON_OUTPUT_DIR}/reconciliation_results.csv"
summary_path = f"{RECON_OUTPUT_DIR}/reconciliation_summary.csv"

validated_extractions_df.to_csv('/tmp/validated_extractions.csv', index=False)
recon_df.to_csv('/tmp/reconciliation_results.csv', index=False)
summary_df = recon_df.groupby(['document_key', 'source_format', 'final_status']).size().reset_index(name='field_count')
summary_df.to_csv('/tmp/reconciliation_summary.csv', index=False)

for local_path, dest_path in [
    ('/tmp/validated_extractions.csv', validated_extractions_path),
    ('/tmp/reconciliation_results.csv', reconciliation_path),
    ('/tmp/reconciliation_summary.csv', summary_path),
]:
    with open(local_path, 'rb') as src, open(dest_path, 'wb') as dst:
        dst.write(src.read())

print('Saved outputs:')
print(validated_extractions_path)
print(reconciliation_path)
print(summary_path)
display(summary_df)

globals()['validated_extractions_df'] = validated_extractions_df
globals()['recon_df'] = recon_df
globals()['summary_df'] = summary_df

Saved outputs:
/Volumes/vabtp_dev/dbfs_files/vabtp_dev/PDF_EXTRACTION/Assignment/Reconciliation-Outputs/validated_extractions.csv
/Volumes/vabtp_dev/dbfs_files/vabtp_dev/PDF_EXTRACTION/Assignment/Reconciliation-Outputs/reconciliation_results.csv
/Volumes/vabtp_dev/dbfs_files/vabtp_dev/PDF_EXTRACTION/Assignment/Reconciliation-Outputs/reconciliation_summary.csv


document_key,source_format,final_status,field_count
genel,csv,False Alert,14
genel,csv,Match,74
genel,csv,True Mismatch,7
genel,json,False Alert,19
genel,json,Match,69
genel,json,True Mismatch,7
idbi,csv,False Alert,10
idbi,csv,Match,44
idbi,csv,True Mismatch,41
idbi,json,False Alert,9


## Submission and repository guidance

Following your preferences, use the same workspace path `/Users/deepak.yadav@contractor.p66.com` as the staging area for repository-ready documentation.

### Saved documentation files

* `README.md`
* `REPORT.md`

### Recommended repository contents

* Notebook export of this workflow
* Any extracted Python source modules if you split logic out of the notebook
* Sample term sheets
* Sample booking CSV and JSON files
* `README.md`
* `REPORT.md`
* `requirements.txt`
* `.env.example`
* Optional example outputs if submission rules allow them

### End-to-end execution order

1. Install dependencies
2. Configure input and output paths
3. Parse PDFs with `ai_parse_document`
4. Export detected tables to Excel
5. Run first-pass field extraction with `ai_query`
6. Normalize and schema-validate extracted payloads
7. Ingest booking CSV and JSON files
8. Reconcile all target fields
9. Run second-pass mismatch adjudication
10. Save final CSV outputs

### Generated outputs from this notebook

* Excel extracts under `Term-Sheets-Extracts`
* `validated_extractions.csv`
* `reconciliation_results.csv`
* `reconciliation_summary.csv`

### GitHub publishing steps

1. Create a public GitHub repository
2. Add the notebook, documentation, sample inputs, and dependency files
3. Commit and push all artifacts
4. Verify a clean clone can run the workflow from the README steps
